In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display, Markdown

# Estilo visual
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

print('Librerías cargadas correctamente')

In [31]:
nRowsRead = None  # Cambia a None si quieres leer todo el archivo

df1 = pd.read_csv('finanzas.csv', delimiter=',', nrows=nRowsRead)
df1.dataframeName = 'finanzas.csv'
nRow, nCol = df1.shape

print(f'There are {nRow} rows and {nCol} columns')
df1.head()   # Muestra las primeras filas

There are 6362620 rows and 11 columns


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [32]:
# Reporte de dimensiones y información del dataset
print("=== REPORTE DEL DATASET ===")
print(f"Número de filas:    {nRow:,}")
print(f"Número de columnas: {nCol}")
print(f"Tamaño total:       {nRow} filas × {nCol} columnas\n")

# Tamaño en memoria
memory_usage = df1.memory_usage(deep=True).sum()
print(f"Tamaño en memoria:  {memory_usage / 1024**2:.2f} MB\n")

# Tipos de datos de cada columna
print("Columnas y tipos de datos:")
print(df1.dtypes)

=== REPORTE DEL DATASET ===
Número de filas:    6,362,620
Número de columnas: 11
Tamaño total:       6362620 filas × 11 columnas

Tamaño en memoria:  1452.57 MB

Columnas y tipos de datos:
step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object


# muestreo

In [33]:
print(f"Dataset cargado: {nRow:,} filas y {nCol} columnas\n")

# Extraer muestra si es necesario
if nRow > 100_000:
    df = df1.sample(n=100_000, random_state=42).reset_index(drop=True)
    print(f"→ Se ha extraído una muestra aleatoria de 100,000 registros (random_state=42)")
else:
    df = df1.copy()
    print("→ El dataset tiene 100,000 filas o menos, se utiliza completo.")

print(f"Dataset final para trabajar: {df.shape[0]:,} filas y {df.shape[1]} columnas")

Dataset cargado: 6,362,620 filas y 11 columnas

→ Se ha extraído una muestra aleatoria de 100,000 registros (random_state=42)
Dataset final para trabajar: 100,000 filas y 11 columnas


# ====================== MUESTREO ESTRATÉGICO ======================

In [34]:
# Reporte de valores nulos
nulos = df.isnull().sum()
porcentaje_nulos = (nulos / len(df) * 100).round(4)

reporte_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': porcentaje_nulos
}).sort_values('Nulos', ascending=False)

print(reporte_nulos)

# analisis de duplicados
porcentaje_dup = (duplicados / len(df) * 100).round(4)

print(f"Filas duplicadas exactas: {duplicados:,}")
print(f"Porcentaje del total: {porcentaje_dup}%")
    
print(f"Dimensiones después de eliminar duplicados: {df.shape[0]:,} filas")

                Nulos  Porcentaje (%)
step                0             0.0
type                0             0.0
amount              0             0.0
nameOrig            0             0.0
oldbalanceOrg       0             0.0
newbalanceOrig      0             0.0
nameDest            0             0.0
oldbalanceDest      0             0.0
newbalanceDest      0             0.0
isFraud             0             0.0
isFlaggedFraud      0             0.0
Filas duplicadas exactas: 0
Porcentaje del total: 0.0%
Dimensiones después de eliminar duplicados: 100,000 filas


In [1]:
print("Tipos de datos actuales:")
print(df.dtypes)

# Identificar posibles problemas
print("\nAnálisis de tipos:")
# Variables que deberían ser numéricas pero están como object
for col in df.columns:
    if df[col].dtype == 'object':
        try:
            # Intentar convertir a numérico
            pd.to_numeric(df[col], errors='coerce')
            print(f"• {col}: Es texto pero parece numérico → Posible conversión recomendada")
        except:
            pass
# Conversión explícita si es necesario (ajusta según tu dataset)
# Ejemplo: Si 'step' debería ser entero

# Optimización de tipos (recomendado)
df['step'] = df['step'].astype('int32')
df['isFraud'] = df['isFraud'].astype('int8')
df['isFlaggedFraud'] = df['isFlaggedFraud'].astype('int8')

print("\nTipos de datos después de conversiones:")
print(df.dtypes)

Tipos de datos actuales:


NameError: name 'df' is not defined

In [42]:
# 1. Montos negativos (imposible en transacciones)
monto_negativo = (df['amount'] < 0).sum()
print(f"1. Montos negativos (amount < 0): {monto_negativo:,} casos")

# 2. Balances negativos en origen (puede ser posible en sobregiro, pero raro)
oldbalance_neg = (df['oldbalanceOrg'] < 0).sum()
newbalance_neg = (df['newbalanceOrig'] < 0).sum()
print(f"2. oldbalanceOrg negativo     : {oldbalance_neg:,} casos")
print(f"3. newbalanceOrig negativo    : {newbalance_neg:,} casos")

# 3. Balances negativos en destino
oldbalance_dest_neg = (df['oldbalanceDest'] < 0).sum()
newbalance_dest_neg = (df['newbalanceDest'] < 0).sum()
print(f"4. oldbalanceDest negativo    : {oldbalance_dest_neg:,} casos")
print(f"5. newbalanceDest negativo    : {newbalance_dest_neg:,} casos")

# 4. Inconsistencias matemáticas (muy importante)
# newbalanceOrig debería ser ≈ oldbalanceOrg - amount (en la mayoría de tipos)
df['diff_orig'] = df['oldbalanceOrg'] - df['newbalanceOrig']
inconsistencia_orig = (abs(df['diff_orig'] - df['amount']) > 1).sum()  # tolerancia de 1 unidad

print(f"6. Inconsistencias en balance origen (old - new != amount): {inconsistencia_orig:,} casos")

# 5. Resumen por tipo de transacción
print("\n--- Inconsistencias por tipo de transacción ---")
inconsistencias_por_tipo = df.groupby('type').apply(
    lambda x: (abs(x['oldbalanceOrg'] - x['newbalanceOrig'] - x['amount']) > 1).sum()
)
print(inconsistencias_por_tipo)

1. Montos negativos (amount < 0): 0 casos
2. oldbalanceOrg negativo     : 0 casos
3. newbalanceOrig negativo    : 0 casos
4. oldbalanceDest negativo    : 0 casos
5. newbalanceDest negativo    : 0 casos
6. Inconsistencias en balance origen (old - new != amount): 78,811 casos

--- Inconsistencias por tipo de transacción ---
type
CASH_IN     22141
CASH_OUT    31355
DEBIT         185
PAYMENT     17181
TRANSFER     7949
dtype: int64


In [45]:
# Crear tabla resumen de calidad
resumen_calidad = pd.DataFrame({
    'Variable': df.columns,
    'Tipo': df.dtypes.values,
    'Nulos (%)': (df.isnull().sum() / len(df) * 100).round(4),
    'Inconsistencias': [0]*len(df.columns)  # Se actualiza manualmente
})

# Actualizar inconsistencias específicas
resumen_calidad.loc[resumen_calidad['Variable'].isin(['oldbalanceOrg', 'newbalanceOrig']), 'Inconsistencias'] = 78811

print(resumen_calidad.to_string(index=False))

      Variable    Tipo  Nulos (%)  Inconsistencias
          step   int32        0.0                0
          type     str        0.0                0
        amount float64        0.0                0
      nameOrig     str        0.0                0
 oldbalanceOrg float64        0.0            78811
newbalanceOrig float64        0.0            78811
      nameDest     str        0.0                0
oldbalanceDest float64        0.0                0
newbalanceDest float64        0.0                0
       isFraud    int8        0.0                0
isFlaggedFraud    int8        0.0                0
     diff_orig float64        0.0                0


#parte estadistica